# Analiza wyników skanowania licencji

Notebook analizuje wyniki z katalogu `wyniki/` — dane JSONL ze skanera zależności.

- `licenses.jsonl` — skan tego projektu (depth 1)
- `licenses_pnw.jsonl` — skan projektu AbsoluteCinema (depth max)

In [ ]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

WYNIKI = Path("../wyniki")

COLORS = {
    "Permissive": "#4caf50",
    "Copyleft": "#f44336",
    "Weak Copyleft": "#ff9800",
    "Proprietary": "#9c27b0",
    "Unknown": "#9e9e9e",
}

def load_jsonl(path: Path) -> pd.DataFrame:
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return pd.DataFrame(records)

df_local = load_jsonl(WYNIKI / "licenses.jsonl")
df_pnw = load_jsonl(WYNIKI / "licenses_pnw.jsonl")

print(f"Local project : {len(df_local)} packages")
print(f"AbsoluteCinema: {len(df_pnw)} packages")

## Rozkład kategorii licencji — wykresy kołowe (porównanie)

In [ ]:
def pie_chart(ax, df, title):
    cat_counts = df["category"].value_counts()
    colors = [COLORS.get(c, "#9e9e9e") for c in cat_counts.index]
    total = len(df)

    wedges, texts, autotexts = ax.pie(
        cat_counts.values,
        colors=colors,
        autopct=lambda p: f"{p:.0f}%" if p >= 5 else "",
        startangle=90,
        textprops={"fontsize": 10},
        pctdistance=0.75,
    )
    # Legend instead of inline labels — avoids overlap on small slices
    legend_labels = [f"{cat} — {n} ({n/total*100:.0f}%)" for cat, n in zip(cat_counts.index, cat_counts.values)]
    ax.legend(wedges, legend_labels, loc="lower center", bbox_to_anchor=(0.5, -0.15), fontsize=9)
    ax.set_title(title, fontsize=13, fontweight="bold")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 8))
pie_chart(ax1, df_local, f"Local project ({len(df_local)} pkgs)")
pie_chart(ax2, df_pnw, f"AbsoluteCinema ({len(df_pnw)} pkgs)")
plt.tight_layout()
plt.savefig(WYNIKI / "categories_pie.png", dpi=150, bbox_inches="tight")
plt.show()

## Packages with risk — AbsoluteCinema (non-permissive)

In [ ]:
risky = df_pnw[df_pnw["category"] != "Permissive"].sort_values("category")
if risky.empty:
    print("All packages are permissive — no risks detected.")
else:
    print(f"{len(risky)} packages with non-permissive or unknown licenses:\n")
    print(risky[["package", "license", "depth", "category"]].to_string(index=False))

## Direct vs transitive dependencies — AbsoluteCinema

In [ ]:
df_pnw["dep_type"] = df_pnw["depth"].apply(lambda d: "Direct (depth 1)" if d == 1 else "Transitive (depth 2+)")

ct = pd.crosstab(df_pnw["category"], df_pnw["dep_type"])
ct = ct.reindex(["Permissive", "Weak Copyleft", "Copyleft", "Proprietary", "Unknown"])
ct = ct.dropna(how="all")

colors_bar = ["#2196f3", "#ff9800"]
ax = ct.plot.barh(stacked=True, color=colors_bar, figsize=(9, 4))
ax.set_xlabel("Number of packages")
ax.set_title("License categories: direct vs transitive dependencies", fontweight="bold")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(WYNIKI / "direct_vs_transitive.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary statistics — porównanie

In [ ]:
def summary(df, name):
    total = len(df)
    permissive = (df["category"] == "Permissive").sum()
    copyleft = df["category"].isin(["Copyleft", "Weak Copyleft"]).sum()
    unknown = (df["category"] == "Unknown").sum()
    print(f"=== {name} ===")
    print(f"  Total packages         : {total}")
    print(f"  Permissive             : {permissive} ({permissive/total*100:.0f}%)")
    print(f"  Copyleft / Weak Copyleft: {copyleft} ({copyleft/total*100:.0f}%)")
    print(f"  Unknown                : {unknown} ({unknown/total*100:.0f}%)")
    print(f"  Risk score: {(copyleft + unknown) / total * 100:.1f}% require attention\n")

summary(df_local, "Local project")
summary(df_pnw, "AbsoluteCinema")